# 📁 Proyecto de Unidad 2: Verificación Estadística de la Calidad de los Datos

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Curso:** Especialización en Aprendizaje Automático: Gestión de Datos y Entrenamiento  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos

---

| Campo | Completa aquí |
|-------|---------------|
| **Nombre del alumno/a** | *(escribe tu nombre)* |
| **Dataset asignado** | *(nombre del dataset)* |
| **Fuente del dataset** | *(URL o referencia)* |
| **Fecha de entrega** | *(fecha)* |

> Las normas de entrega, formato y defensa oral son las mismas que en el proyecto de UD1. Nombre del fichero: `UD2_proyecto_APELLIDO_NOMBRE.ipynb`

In [ ]:
# ── Importaciones ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (shapiro, pearsonr, spearmanr, chi2_contingency,
                          ttest_ind, mannwhitneyu, f_oneway, kruskal, levene)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

# ── Carga del dataset ──────────────────────────────────────────────────────
# TU CÓDIGO AQUÍ
# df = pd.read_csv('...')
# print(df.shape)
# df.head()

---
## Sección 1 — Estadística descriptiva
*(Bloque 1: centralidad, dispersión y detección de baja varianza)*

**Objetivo:** caracterizar numéricamente las variables del dataset. Al terminar esta sección debes saber qué rango, dispersión y forma tiene cada variable, y cuáles tienen tan poca variación que no aportarán información a un modelo.

### 1.1 Medidas de centralidad

Calcula media, mediana y moda para cada variable numérica. Identifica aquellas donde la diferencia entre media y mediana sea considerable.

In [ ]:
# TU CÓDIGO AQUÍ
# Identifica las columnas numéricas
# cols_num = df.select_dtypes(include='number').columns.tolist()

# Calcula media, mediana y moda para cada una
# resumen = pd.DataFrame({
#     'Media':   df[cols_num].mean(),
#     'Mediana': df[cols_num].median(),
#     'Moda':    df[cols_num].mode().iloc[0],
# })
# resumen['Media - Mediana'] = (resumen['Media'] - resumen['Mediana']).round(3)
# print(resumen.round(3))

**📝 Conclusión 1.1** *(obligatorio)*

¿En qué variables hay una diferencia notable entre media y mediana? ¿Qué te indica eso sobre su distribución? ¿Qué medida de centralidad usarías para describir cada variable y por qué?

*Escribe tu respuesta aquí:*

### 1.2 Medidas de dispersión

Calcula la dispersión de cada variable numérica. Usa el coeficiente de variación (CV) para comparar variables con escalas distintas e identifica las que tienen baja varianza.

In [ ]:
# TU CÓDIGO AQUÍ
# resumen_disp = pd.DataFrame({
#     'Std':       df[cols_num].std(),
#     'IQR':       df[cols_num].quantile(0.75) - df[cols_num].quantile(0.25),
#     'CV (%)':    (df[cols_num].std() / df[cols_num].mean().abs() * 100).round(2),
#     'Rango':     df[cols_num].max() - df[cols_num].min(),
# })
# resumen_disp['Baja varianza'] = resumen_disp['CV (%)'] < 5
# print(resumen_disp.round(3))

In [ ]:
# TU CÓDIGO AQUÍ: visualización de dispersión
# Boxplot conjunto para comparar la dispersión relativa de las variables numéricas
# (normaliza primero si las escalas son muy distintas)

# from sklearn.preprocessing import StandardScaler
# df_norm = pd.DataFrame(StandardScaler().fit_transform(df[cols_num]),
#                         columns=cols_num)
# fig, ax = plt.subplots(figsize=(12, 4))
# df_norm.boxplot(ax=ax)
# ax.set_title('Dispersión relativa (datos estandarizados)')
# ax.axhline(0, color='red', linestyle='--', alpha=0.5)
# plt.xticks(rotation=30)
# plt.tight_layout()
# plt.show()

**📝 Conclusión 1.2** *(obligatorio)*

¿Qué variable tiene la dispersión relativa más alta? ¿Y la más baja? ¿Hay alguna variable con CV < 5% que podrías descartar por baja varianza? ¿Qué implicaciones tendría incluir variables de muy baja varianza en un modelo?

*Escribe tu respuesta aquí:*

---
## Sección 2 — Distribuciones y normalidad
*(Bloque 2: forma de la distribución, sesgo, curtosis, test de normalidad y transformaciones)*

**Objetivo:** conocer la forma de la distribución de cada variable numérica, evaluar si se ajusta a la normalidad y determinar si es necesario aplicar transformaciones antes del modelado.

### 2.1 Forma de las distribuciones

Para cada variable numérica, calcula el sesgo (skewness) y la curtosis, y visualiza el histograma con la curva KDE superpuesta.

In [ ]:
# TU CÓDIGO AQUÍ
# resumen_forma = pd.DataFrame({
#     'Sesgo':    df[cols_num].skew().round(3),
#     'Curtosis': df[cols_num].kurt().round(3),
# })
# resumen_forma['Alerta sesgo'] = resumen_forma['Sesgo'].abs().apply(
#     lambda s: '⚠️ Alta' if s > 1 else ('〰️ Moderada' if s > 0.5 else '✅ Simétrica')
# )
# print(resumen_forma)

In [ ]:
# TU CÓDIGO AQUÍ: histograma + KDE para las variables con mayor sesgo
# Representa al menos las 3 variables con mayor |sesgo|

# Ejemplo para una variable:
# fig, ax = plt.subplots(figsize=(8, 4))
# serie = df['col'].dropna()
# ax.hist(serie, bins=30, density=True, color='steelblue',
#         edgecolor='white', alpha=0.6, label='Histograma')
# serie.plot.kde(ax=ax, color='crimson', linewidth=2, label='KDE')
# ax.set_title(f'col — sesgo={serie.skew():.3f}')
# ax.legend()
# plt.tight_layout()
# plt.show()

**📝 Conclusión 2.1** *(obligatorio)*

¿Qué variables presentan sesgo positivo? ¿Y negativo? ¿Alguna tiene una curtosis muy alta (distribución puntiaguda) o muy baja (achatada)? ¿Qué tipo de distribución teórica (normal, uniforme, exponencial…) se aproxima más a cada una?

*Escribe tu respuesta aquí:*

### 2.2 Test de normalidad (Shapiro-Wilk)

Aplica el test de Shapiro-Wilk a cada variable numérica. Complementa el resultado numérico con un gráfico Q-Q.

In [ ]:
# TU CÓDIGO AQUÍ
# print(f"{'Variable':<25} {'W':>8} {'p-valor':>12} {'Normal (α=0.05)?':>20}")
# print('─' * 68)
# for col in cols_num:
#     serie = df[col].dropna()
#     muestra = serie if len(serie) <= 5000 else serie.sample(5000, random_state=42)
#     W, p = shapiro(muestra)
#     estado = '✅ Sí' if p > 0.05 else '❌ No'
#     print(f'{col:<25} {W:>8.4f} {p:>12.6f} {estado:>20}')

In [ ]:
# TU CÓDIGO AQUÍ: gráfico Q-Q para las 2 variables más relevantes
# (elige una que pase el test y otra que no, si las hay)

# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# for ax, col in zip(axes, [col_normal, col_no_normal]):
#     stats.probplot(df[col].dropna(), dist='norm', plot=ax)
#     ax.set_title(f'Q-Q plot — {col}')
# plt.tight_layout()
# plt.show()

**📝 Conclusión 2.2** *(obligatorio)*

¿Qué variables pasan el test de Shapiro-Wilk? ¿Coincide el resultado del test con lo que observabas en los histogramas? ¿Para cuáles sería más importante aplicar una transformación antes de usar métodos que asumen normalidad?

*Escribe tu respuesta aquí:*

### 2.3 Transformaciones

Para las variables con sesgo alto que no siguen una distribución normal, aplica al menos una transformación (log1p, Box-Cox o Yeo-Johnson) y compara la distribución antes y después.

In [ ]:
# TU CÓDIGO AQUÍ: elige la variable con más sesgo positivo y aplica log1p
# col_sesgada = '...'
# df['col_log'] = np.log1p(df[col_sesgada])

# Compara antes y después
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].hist(df[col_sesgada].dropna(), bins=30, color='coral', edgecolor='white')
# axes[0].set_title(f'Original — sesgo={df[col_sesgada].skew():.3f}')
# axes[1].hist(df['col_log'].dropna(), bins=30, color='mediumseagreen', edgecolor='white')
# axes[1].set_title(f'Tras log1p — sesgo={df["col_log"].skew():.3f}')
# plt.tight_layout()
# plt.show()

**📝 Conclusión 2.3** *(obligatorio)*

¿Ha mejorado la simetría tras la transformación? ¿Pasaría ahora el test de Shapiro-Wilk? ¿Aplicarías esta transformación de cara al modelado? Justifica.

*Escribe tu respuesta aquí:*

---
## Sección 3 — Correlación y selección de variables
*(Bloque 3: Pearson, Spearman, Cramér's V, significación, multicolinealidad)*

**Objetivo:** cuantificar las relaciones entre variables y determinar cuáles son relevantes para el objetivo del análisis y cuáles son redundantes entre sí.

### 3.1 Correlación entre variables numéricas

Calcula y visualiza las matrices de correlación de Pearson y Spearman. Identifica los pares con correlación más alta y los más bajos.

In [ ]:
# TU CÓDIGO AQUÍ
# r_pearson  = df[cols_num].corr(method='pearson')
# r_spearman = df[cols_num].corr(method='spearman')

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# for ax, mat, titulo in zip(axes,
#                             [r_pearson, r_spearman],
#                             ['Pearson', 'Spearman']):
#     sns.heatmap(mat, annot=True, fmt='.2f', cmap='coolwarm',
#                 center=0, vmin=-1, vmax=1,
#                 linewidths=0.5, linecolor='white', ax=ax, square=True)
#     ax.set_title(f'Correlación de {titulo}')
# plt.tight_layout()
# plt.show()

In [ ]:
# TU CÓDIGO AQUÍ: calcula r y p-valor para el par más correlacionado
# r, p = pearsonr(df['col_x'].dropna(), df['col_y'].dropna())
# print(f'Pearson r={r:.4f}, p={p:.6f}')

# Intervalo de confianza al 95% (z de Fisher)
# n = df[['col_x', 'col_y']].dropna().shape[0]
# z = np.arctanh(r)
# se = 1 / np.sqrt(n - 3)
# z_crit = stats.norm.ppf(0.975)
# lo, hi = np.tanh(z - z_crit*se), np.tanh(z + z_crit*se)
# print(f'IC 95%: [{lo:.4f}, {hi:.4f}]')

**📝 Conclusión 3.1** *(obligatorio)*

¿Qué par de variables tiene la correlación más alta? ¿Y la más baja? ¿Hay diferencias notables entre Pearson y Spearman en algún par? ¿Qué te indica eso sobre la linealidad de esa relación? ¿La correlación más alta es estadísticamente significativa?

*Escribe tu respuesta aquí:*

### 3.2 Asociación entre variables categóricas (Cramér's V)

Si tu dataset tiene variables categóricas, calcula la matriz de Cramér's V. Si no las tiene, indica aquí que este apartado no aplica y justifica brevemente.

In [ ]:
# TU CÓDIGO AQUÍ (omite si no hay variables categóricas)
# cols_cat = df.select_dtypes(include=['object', 'category']).columns.tolist()

# def cramers_v(col1, col2):
#     tabla = pd.crosstab(col1, col2)
#     chi2, p, dof, _ = chi2_contingency(tabla)
#     n = tabla.sum().sum()
#     return np.sqrt(chi2 / (n * (min(tabla.shape) - 1))), p

# n = len(cols_cat)
# matrix = pd.DataFrame(np.zeros((n, n)), index=cols_cat, columns=cols_cat)
# for i, c1 in enumerate(cols_cat):
#     for j, c2 in enumerate(cols_cat):
#         if i == j: matrix.loc[c1, c2] = 1.0
#         elif i < j:
#             V, _ = cramers_v(df[c1].dropna(), df[c2].dropna())
#             matrix.loc[c1, c2] = matrix.loc[c2, c1] = V

# fig, ax = plt.subplots(figsize=(7, 5))
# sns.heatmap(matrix, annot=True, fmt='.3f', cmap='YlOrRd',
#             vmin=0, vmax=1, linewidths=0.5, ax=ax, square=True)
# ax.set_title("Matriz de Cramér's V")
# plt.tight_layout()
# plt.show()

### 3.3 Multicolinealidad y selección de variables

Identifica pares de variables predictoras con correlación alta (|r| > 0.85). Propón cuál descartarías de cada par y justifica la elección.

In [ ]:
# TU CÓDIGO AQUÍ
# Identifica pares con alta correlación
# umbral = 0.85
# print(f'Pares con |r| > {umbral}:')
# for i, c1 in enumerate(cols_num):
#     for j, c2 in enumerate(cols_num):
#         if j <= i: continue
#         r = r_pearson.loc[c1, c2]
#         if abs(r) > umbral:
#             print(f'  {c1} vs {c2}: r = {r:.3f}')

**📝 Conclusión 3.3** *(obligatorio)*

Para cada par con multicolinealidad detectada: ¿cuál de las dos variables conservarías y por qué (mayor relevancia con el objetivo, mayor interpretabilidad, menos nulos…)? ¿Cuántas variables quedarían tras aplicar el criterio de multicolinealidad?

*Escribe tu respuesta aquí:*

---
## Sección 4 — Tests de independencia
*(Bloque 4: chi-cuadrado, t-test, Mann-Whitney, ANOVA, Kruskal-Wallis, Mutual Information)*

**Objetivo:** verificar estadísticamente si las relaciones observadas en la sección anterior son significativas, y cuantificar la relevancia de cada variable respecto al objetivo.

### 4.1 Chi-cuadrado (variables categóricas)

Si tu dataset tiene una variable objetivo categórica y otras variables categóricas, aplica el test chi-cuadrado para evaluar su independencia. Si no aplica, indícalo y justifica.

In [ ]:
# TU CÓDIGO AQUÍ
# col_objetivo_cat = '...'   # variable objetivo categórica
# cols_cat_pred = [...]      # predictoras categóricas

# print(f"{'Variable':<25} {'chi2':>8} {'p-valor':>12} {"Cramér V":>10} {'Significativa':>15}")
# print('─' * 73)
# for col in cols_cat_pred:
#     tmp = df[[col, col_objetivo_cat]].dropna()
#     tabla = pd.crosstab(tmp[col], tmp[col_objetivo_cat])
#     chi2, p, dof, _ = chi2_contingency(tabla)
#     n = tabla.sum().sum()
#     V = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
#     sig = '✅ Sí' if p < 0.05 else '❌ No'
#     print(f'{col:<25} {chi2:>8.3f} {p:>12.6f} {V:>10.4f} {sig:>15}')

**📝 Conclusión 4.1** *(obligatorio, o indicar que no aplica)*

¿Qué variables categóricas tienen una asociación significativa con el objetivo? ¿Cuál tiene el Cramér's V más alto? ¿Hay alguna que sea estadísticamente significativa pero con V muy bajo (significativa pero irrelevante en la práctica)?

*Escribe tu respuesta aquí:*

### 4.2 Comparación de grupos (t-test / Mann-Whitney / ANOVA)

Elige al menos dos variables numéricas relevantes y compara su distribución entre los grupos definidos por la variable objetivo (u otra variable categórica significativa). Sigue el árbol de decisión: verifica normalidad y homocedasticidad antes de elegir el test.

In [ ]:
# TU CÓDIGO AQUÍ: árbol de decisión para elegir el test correcto
# col_num   = '...'   # variable numérica a comparar
# col_grupo = '...'   # variable de agrupación

# grupos_unicos = df[col_grupo].dropna().unique()
# print(f'Grupos detectados: {grupos_unicos}')
# print()

# ─ Paso 1: normalidad por grupo
# for g in grupos_unicos:
#     serie = df.loc[df[col_grupo] == g, col_num].dropna()
#     muestra = serie if len(serie) <= 5000 else serie.sample(5000)
#     _, p_n = shapiro(muestra)
#     print(f'  Grupo {g}: n={len(serie)}, Shapiro p={p_n:.4f}  {"✅" if p_n > 0.05 else "⚠️"}')

# ─ Paso 2: homocedasticidad (si hay normalidad)
# grupos_data = [df.loc[df[col_grupo] == g, col_num].dropna() for g in grupos_unicos]
# _, p_lev = levene(*grupos_data)
# print(f'\nLevene p={p_lev:.4f}  {"✅ Homocedástico" if p_lev > 0.05 else "⚠️ Heterocedástico"}')

# ─ Paso 3: test según condiciones
# if 2 grupos → ttest_ind o mannwhitneyu
# si > 2 grupos → f_oneway o kruskal

In [ ]:
# TU CÓDIGO AQUÍ: visualización de la comparación
# fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# sns.boxplot(data=df, x=col_grupo, y=col_num, ax=axes[0])
# axes[0].set_title(f'{col_num} por {col_grupo}')
# for g in grupos_unicos:
#     serie = df.loc[df[col_grupo] == g, col_num].dropna()
#     serie.plot.kde(ax=axes[1], label=str(g))
# axes[1].set_title('Distribución por grupo')
# axes[1].legend()
# plt.tight_layout()
# plt.show()

**📝 Conclusión 4.2** *(obligatorio)*

¿Qué test has aplicado y por qué (justifica el árbol de decisión)? ¿Los grupos tienen distribuciones significativamente distintas? Si usaste ANOVA o Kruskal-Wallis, ¿qué pares concretos difieren (Tukey o inspección visual)? ¿Qué tamaño tiene el efecto?

*Escribe tu respuesta aquí:*

### 4.3 Mutual Information

Calcula la Mutual Information entre cada variable predictora y la variable objetivo. Compara los resultados con la correlación de Pearson obtenida en la sección anterior.

In [ ]:
# TU CÓDIGO AQUÍ
# col_objetivo = '...'   # variable objetivo (numérica o categórica)

# Prepara X e y sin nulos
# cols_pred = [c for c in cols_num if c != col_objetivo]
# datos_mi = df[cols_pred + [col_objetivo]].dropna()
# X_mi = datos_mi[cols_pred]
# y_mi = datos_mi[col_objetivo]

# Elige la función según el tipo de objetivo:
# mi = mutual_info_regression(X_mi, y_mi, random_state=42)    # si y es continua
# mi = mutual_info_classif(X_mi, y_mi, random_state=42)       # si y es categórica

# mi_df = pd.DataFrame({'Variable': cols_pred, 'MI': mi}).sort_values('MI', ascending=False)
# print(mi_df.to_string(index=False))

# Visualización
# fig, ax = plt.subplots(figsize=(9, 4))
# ax.barh(mi_df['Variable'], mi_df['MI'], color='steelblue', edgecolor='white')
# ax.axvline(0.01, color='crimson', linestyle='--', label='Umbral 0.01')
# ax.set_xlabel('Mutual Information')
# ax.set_title(f"MI respecto a '{col_objetivo}'")
# ax.legend()
# plt.tight_layout()
# plt.show()

**📝 Conclusión 4.3** *(obligatorio)*

¿Qué variable tiene la MI más alta? ¿Coincide con la de mayor correlación de Pearson? Si hay discrepancias (alta MI, baja correlación o viceversa), ¿qué podría explicarlas? ¿Descartarías alguna variable por MI < 0.01?

*Escribe tu respuesta aquí:*

---
## Sección 5 — Informe de verificación estadística

**Objetivo:** sintetizar todo el análisis en un documento técnico que oriente las decisiones de diseño del modelo.

### 5.1 Tabla resumen de variables

Completa la siguiente tabla para cada variable del dataset:

| Variable | Tipo | Sesgo | Normal | Outliers | Relevancia (MI) | Multicolineal con | Decisión |
|----------|------|-------|--------|----------|-----------------|-------------------|----------|
| *col_1* | *continua* | *alta (+)* | *No* | *3%* | *Alta (0.32)* | *col_3* | *Transformar (log1p)* |
| *col_2* | *categórica* | *—* | *—* | *—* | *Media (0.12)* | *—* | *Incluir* |
| *col_3* | *continua* | *baja* | *Sí* | *1%* | *Alta (0.29)* | *col_1* | *Descartar (redundante)* |
| ... | ... | ... | ... | ... | ... | ... | ... |

*(edita esta tabla con tus variables reales)*

### 5.2 Recomendaciones de diseño

**📝 Escribe aquí un texto de 10-15 líneas** dirigido a quien vaya a entrenar el modelo con este dataset. Debe cubrir:

1. **Variables a incluir** y en qué forma (original, transformada, codificada)
2. **Variables a descartar** con justificación estadística concreta
3. **Tratamientos pendientes** antes del entrenamiento (imputación, normalización, codificación de categorías)
4. **Riesgos o limitaciones** del dataset que podrían afectar al modelo (sesgo de representación, volumen insuficiente, alta correlación residual…)

*Escribe tu respuesta aquí:*

### 5.3 Reflexión final

**📝 Responde brevemente:**

1. ¿Qué test o técnica de esta unidad te ha resultado más útil para este dataset y por qué?
2. ¿Ha habido algún resultado que te haya sorprendido o contradicho tu intuición inicial?
3. Si tuvieras que repetir este análisis con más tiempo, ¿qué añadirías?

*Escribe tu respuesta aquí:*

---
## 📋 Rúbrica de evaluación

> Esta sección es informativa. No la modifiques.

### Cuaderno (70% de la nota del proyecto)

| Criterio | Insuficiente (0-4) | Suficiente (5-6) | Notable (7-8) | Sobresaliente (9-10) |
|----------|--------------------|-----------------|---------------|---------------------|
| **Estadística descriptiva** (15%) | Solo `describe()` sin interpretación | CV e IQR calculados | Dispersión + detección baja varianza comentada | Análisis completo con decisión justificada sobre cada variable |
| **Distribuciones y normalidad** (15%) | Sin análisis de forma | Sesgo calculado | Sesgo + Shapiro-Wilk + Q-Q | Todo lo anterior + transformación aplicada y evaluada |
| **Correlación** (20%) | Sin análisis de correlación | Mapa de Pearson | Pearson + Spearman + IC del par más relevante | Todo lo anterior + Cramér's V (si aplica) + análisis de multicolinealidad |
| **Tests de independencia** (20%) | Sin tests | Un test aplicado sin justificación | Test correcto según condiciones con visualización | Árbol de decisión completo + MI + comparación MI vs Pearson interpretada |
| **Informe final** (15%) | Sin tabla ni recomendaciones | Tabla parcial | Tabla completa + recomendaciones básicas | Recomendaciones accionables con justificación estadística explícita |
| **Forma y ejecución** (15%) | Errores al ejecutar | Ejecuta con warnings | Ejecuta limpio, código comentado | Código limpio, visualizaciones cuidadas, HTML exportado |

### Defensa oral (30% de la nota del proyecto)

| Criterio | Peso | Descripción |
|----------|------|-------------|
| **Dominio estadístico** | 35% | Explica qué mide cada test, cuándo usarlo y qué supuestos requiere |
| **Justificación de decisiones** | 35% | Argumenta por qué descartó o transformó cada variable |
| **Respuesta a preguntas** | 30% | Responde con rigor y reconoce con honestidad lo que no sabe |

```
Nota proyecto = 0.70 × nota_cuaderno + 0.30 × nota_defensa
```

> ⚠️ **Condición mínima:** el cuaderno debe ejecutarse sin errores (`Kernel → Restart & Run All`) y todas las celdas de conclusión deben estar completadas. Las celdas no aplicables deben indicarlo explícitamente con justificación.